In [8]:
# ==========================================
# Credit Scoring Project Playbook (CSV version)
# ==========================================

# 1️⃣ Импорт библиотек
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------
# 2️⃣ Загрузка данных
df = pd.read_csv("UCI_Credit_Card.csv")

# Проверяем названия колонок
print("Columns:", df.columns)

# Убираем пробелы по краям названий
df.rename(columns=lambda x: x.strip(), inplace=True)

# Переименовываем колонку цели (поддержка разных вариантов названия)
target_candidates = ['default', 'default payment next month', 'default.payment.next.month']
target_column = next((c for c in target_candidates if c in df.columns), None)
if target_column is None:
    raise KeyError(f"Не найдена целевая колонка. Ожидалась одна из: {target_candidates}")
df.rename(columns={target_column: 'default'}, inplace=True)

# Проверка
print("'default' in columns?", 'default' in df.columns)
print(df.head())

# ------------------------------------------
# 3️⃣ Предобработка категориальных признаков
cat_features = ['SEX', 'EDUCATION', 'MARRIAGE']
df[cat_features] = df[cat_features].astype(str)

# One-Hot Encoding
df = pd.get_dummies(df, columns=cat_features, drop_first=True)

# ------------------------------------------
# 4️⃣ Feature Engineering
df['avg_bill_amt'] = df[['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6']].mean(axis=1)
df['avg_pay_amt'] = df[['PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']].mean(axis=1)
df['pay_ratio'] = df['avg_pay_amt'] / (df['avg_bill_amt'] + 1e-5)  # избегаем деление на ноль

# ------------------------------------------
# 5️⃣ Разделение на X и y
y = df['default']
columns_to_drop = [col for col in ['ID', 'default'] if col in df.columns]
X = df.drop(columns=columns_to_drop)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ------------------------------------------
# 6️⃣ Обучение модели
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

# ------------------------------------------
# 7️⃣ Предсказания и оценка
y_pred_proba = clf.predict_proba(X_test)[:,1]  # вероятность дефолта
y_pred = clf.predict(X_test)

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

# ------------------------------------------
# 8️⃣ Важность признаков
feat_importances = pd.Series(clf.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh')
plt.title("Top 10 Feature Importances")
plt.show()

# ------------------------------------------
# 9️⃣ Кредитный рейтинг
X_test = X_test.copy()
X_test['PD'] = y_pred_proba
X_test['Credit_Score'] = (600 - X_test['PD'] * 300).round()
print(X_test[['PD','Credit_Score']].head())

# ------------------------------------------
# 🔟 Визуализация распределения рейтинга
sns.histplot(X_test['Credit_Score'], bins=30, kde=True)
plt.title("Distribution of Credit Scores")
plt.xlabel("Credit Score")
plt.ylabel("Number of Clients")
plt.show()

# ------------------------------------------
# 11️⃣ Сегментация по риску
def risk_category(score):
    if score <= 300:
        return 'High Risk'
    elif score <= 450:
        return 'Medium Risk'
    else:
        return 'Low Risk'

X_test['Risk_Category'] = X_test['Credit_Score'].apply(risk_category)
sns.countplot(x='Risk_Category', data=X_test, order=['Low Risk','Medium Risk','High Risk'])
plt.title("Clients by Risk Category")
plt.show()



Columns: Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default.payment.next.month'],
      dtype='object')
'default' in columns? False
   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1    20000.0    2          2         1   24      2      2     -1     -1   
1   2   120000.0    2          2         2   26     -1      2      0      0   
2   3    90000.0    2          2         2   34      0      0      0      0   
3   4    50000.0    2          2         1   37      0      0      0      0   
4   5    50000.0    1          2         1   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...        0.0        0.0        0.0       0.0     689.0      

NameError: name 'y' is not defined